# Lab 03: Practical Pandas — Sorting, De-duping, GroupBy, Merging, and Column Ops (≈45 min)

**Data file:** `SampleData.csv`

**Goals:** Sorting, de-duplicating, merging, groupby aggregations, and column manipulation.

## 0) Setup & Quick Peek (≈5 min)

In [ ]:
import pandas as pd
from pathlib import Path

# Adjust path if needed
csv_path = Path("SampleData.csv")  # or Path("/mnt/data/SampleData.csv")
df = pd.read_csv(csv_path)

df.head()

In [ ]:
df.shape, df.dtypes

In [ ]:
# Convert Invoice_date to datetime and add helpers
df['Invoice_date'] = pd.to_datetime(df['Invoice_date'])
df['Year']  = df['Invoice_date'].dt.year
df['Month'] = df['Invoice_date'].dt.month

df.describe()

In [ ]:
df.shape, df.dtypes

In [ ]:
df['Account_no'] = df['Account_no'].astype('string')
df.describe(include=['int64', 'float64'])

In [ ]:
df[['Account_no','LocationID','ProductID']].nunique()

## Part A — Sorting & De-duping (≈10–12 min)

**A1) Sort by a single column** — See highest usage first.

In [ ]:
sorted_usage = df.sort_values(by='Billed_usage_kwh', ascending=False)
sorted_usage[['Account_no','LocationID','ProductID','Invoice_date','Billed_usage_kwh']].head(10)

**A2) Multi-column sort** — Chronological within each account.

In [ ]:
sorted_multi = df.sort_values(by=['Account_no','Invoice_date'], ascending=[True, True])
sorted_multi.head(10)

**A3) Detect potential duplicates** — Exact vs. key-based duplicates.

In [ ]:
# Exact duplicate rows (all columns identical)
first_row = df.iloc[[0]]
new_row = pd.DataFrame({'Account_no': first_row['Account_no'],
                         'CustomerID': 'P99',
                         'ProductID': first_row['ProductID'],
                         'Invoice_date': first_row['Invoice_date']})
df = pd.concat([df, new_row], ignore_index=True)
exact_dups_mask = df.duplicated(keep=False)
df[exact_dups_mask].sort_values(df.columns.tolist()).head()

In [ ]:
# Business-key duplicates: same account + date + product
key_cols = ['Account_no','Invoice_date','ProductID']
key_dups_mask = df.duplicated(subset=key_cols, keep=False)
df[key_dups_mask].sort_values(key_cols).head()

**A4) Remove duplicates safely** — Keep the “last” occurrence on key columns.

In [ ]:
df_nodup = df.drop_duplicates(subset=key_cols, keep='last').copy()
print("Before:", len(df), "After:", len(df_nodup))

## Part B — GroupBy & Aggregations (≈10–12 min)

**B1) Product-level usage and revenue summary**

In [ ]:
prod_summary = (
    df_nodup
    .groupby('ProductID', as_index=False)
    .agg(
        total_usage_kwh = ('Billed_usage_kwh','sum'),
        avg_usage_kwh   = ('Billed_usage_kwh','mean'),
        max_usage_kwh   = ('Billed_usage_kwh','max'),
        total_revenue   = ('Bill','sum')
    )
    .sort_values('total_usage_kwh', ascending=False)
)
prod_summary

**B2) Monthly revenue by product (pivot-style)**

In [ ]:
monthly_prod = (
    df_nodup
    .groupby(['Year','Month','ProductID'], as_index=False)
    .agg(monthly_revenue=('Bill','sum'),
         monthly_usage_kwh=('Billed_usage_kwh','sum'))
    .sort_values(['Year','Month','ProductID'])
)
monthly_prod.head(12)

## Part C — Merging with a Lookup Table (≈8–10 min)

Create a tiny in-memory lookup that adds friendly names and attributes per `ProductID`.

In [ ]:
product_lookup = pd.DataFrame({
    'ProductID': ['P01','P02','P03'],
    'ProductName': ['Standard Saver','Peak Flex','Green Choice'],
    'Tier': ['Standard','Peak','Green'],
    'Is_Green': [False, False, True]
})
product_lookup

**C1) Left-merge onto the fact data**

In [ ]:
df_enriched = df_nodup.merge(product_lookup, on='ProductID', how='left')
df_enriched[['ProductID','ProductName','Tier','Is_Green']].drop_duplicates()

**C2) Re-run a summary with the enriched attributes**

In [ ]:
tier_summary = (
    df_enriched
    .groupby('Tier', as_index=False)
    .agg(
        customers=('Account_no','nunique'),
        locations=('LocationID','nunique'),
        total_usage_kwh=('Billed_usage_kwh','sum'),
        total_revenue=('Bill','sum')
    )
    .sort_values('total_revenue', ascending=False)
)
tier_summary

## Part D — Column Manipulation & Basic Functions (≈8–10 min)

**D1) Create calculated columns** — Recompute bill and compare.

In [ ]:
df_enriched['calc_bill'] = df_enriched['Base_charge'] + df_enriched['Price'] * df_enriched['Billed_usage_kwh']
df_enriched['bill_diff'] = (df_enriched['calc_bill'] - df_enriched['Bill']).round(2)

df_enriched['bill_diff'].describe()

In [ ]:
df_enriched.loc[df_enriched['bill_diff'].abs() > 0.01, 
                ['Account_no','Invoice_date','ProductID','Bill','calc_bill','bill_diff']].head()

**D2) Categorize usage with `pd.cut`**

In [ ]:
import numpy as np

bins = [0, 500, 1000, 2000, float('inf')]
labels = ['Low','Moderate','High','Very High']
df_enriched['usage_band'] = pd.cut(df_enriched['Billed_usage_kwh'], bins=bins, labels=labels, right=True)
df_enriched['usage_band'].value_counts()

**D3) Add a boolean flag with `np.where`**

In [ ]:
df_enriched['is_high_usage'] = np.where(df_enriched['Billed_usage_kwh'] >= 1000, True, False)
df_enriched['is_high_usage'].mean()  # proportion

**D4) Simple row-wise function via `apply`** *(vectorized ops are usually faster)*

In [ ]:
def usage_per_dollar(row):
    return row['Billed_usage_kwh'] / row['Bill'] if row['Bill'] else None

df_enriched['kwh_per_dollar'] = df_enriched.apply(usage_per_dollar, axis=1)
df_enriched[['Billed_usage_kwh','Bill','kwh_per_dollar']].head()

## Part E — Export a Clean Result (≈2–3 min)

In [ ]:
monthly_clean = (
    df_enriched
    .groupby(['Year','Month','ProductName'], as_index=False)
    .agg(revenue=('Bill','sum'), usage_kwh=('Billed_usage_kwh','sum'))
    .sort_values(['Year','Month','revenue'], ascending=[True, True, False])
)

monthly_clean.to_csv("monthly_summary.csv", index=False)
print("Wrote monthly_summary.csv")

## Optional Stretch

**1) De-dupe simulation** — Append a duplicate row and verify removal.

In [ ]:
dupe_row = df_enriched.iloc[[0]]
df_dupe_test = pd.concat([df_enriched, dupe_row], ignore_index=True)
len_before = len(df_dupe_test)
df_dupe_test = df_dupe_test.drop_duplicates(subset=['Account_no','Invoice_date','ProductID'], keep='last')
len_after = len(df_dupe_test)
len_before, len_after

**2) Multi-metric aggregation**

In [ ]:
complex_agg = (
    df_enriched
    .groupby('ProductName', as_index=False)
    .agg(
        revenue_sum=('Bill','sum'),
        bill_min=('Bill','min'),
        bill_max=('Bill','max'),
        accounts=('Account_no','nunique')
    )
    .sort_values('revenue_sum', ascending=False)
)
complex_agg